In [1]:
import boto3, io, gzip, os, pandas as pd
import numpy as np
from parking_processing.utils.s3 import parse_s3_uri
import json
from parking_processing.utils.s3 import s3_client, parse_s3_uri, s3_get_text

S3_ROOT = "s3://smart-park-seattle/parking_v2/" #v2
YEAR = 2023

s3 = s3_client()
bucket, prefix = parse_s3_uri(S3_ROOT)
splits = json.loads(s3_get_text(s3, bucket, f"{prefix}splits/year={YEAR}/splits.json"))
weeks = splits["train_weeks"] + splits["val_weeks"] + splits["test_weeks"]

In [2]:
def read_csv_gz_s3(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    raw = obj["Body"].read()
    with gzip.GzipFile(fileobj=io.BytesIO(raw), mode="rb") as f:
        return pd.read_csv(f)

In [3]:
def write_csv_gz_local(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with gzip.open(path, "wt", encoding="utf-8") as f:
        df.to_csv(f, index=False)

In [4]:
def safe_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1-eps)
    return np.log(p/(1-p))

In [5]:
# Merge fallback_preds and mvstgcn_preds

def merge_week(week, target):
    col = "p15" if target == "y_15" else "p30"
    logit_col = "logit15" if target == "y_15" else "logit30"

    base_key = f"{prefix}preds/mvstgcn/year={YEAR}/target={target}/week={week}/pred.csv.gz"
    fb_key   = f"{prefix}preds/fallback/year={YEAR}/target={target}/week={week}/pred.csv.gz"
    out_key  = f"{prefix}preds/final/year={YEAR}/target={target}/week={week}/pred.csv.gz"
    suc_key  = f"{prefix}preds/final/year={YEAR}/target={target}/week={week}/_SUCCESS"

    base = read_csv_gz_s3(bucket, base_key)

    # baseline
    keep_cols = ["ts15_utc", "sourceelementkey", col]
    if logit_col in base.columns:
        keep_cols.append(logit_col)

    merged = base.copy()

    try:
        fb = read_csv_gz_s3(bucket, fb_key)
        merged = merged.merge(fb, on=["ts15_utc", "sourceelementkey"], how="left", suffixes=("", "_fb"))

        # rows with fallback, p = fallback으로 덮기
        use_fb = merged[f"{col}_fb"].notna()
        merged.loc[use_fb, col] = merged.loc[use_fb, f"{col}_fb"]

        # logit 컬럼이 있으면, fallback 적용된 행의 logit도 p에 맞춰 재계산
        if logit_col in merged.columns:
            merged.loc[use_fb, logit_col] = safe_logit(merged.loc[use_fb, col].to_numpy())

    except Exception as e:
        print("merge_week: no/failed fallback:", repr(e))
        # merged는 base 그대로 유지 (logit 있으면 keep_cols에 이미 포함)

    # 최종 컬럼 정리 (logit 포함 여부는 keep_cols가 결정)
    merged = merged[keep_cols]

    local_out = f"/tmp/final_{YEAR}_{week}_{target}.csv.gz"
    write_csv_gz_local(merged, local_out)

    s3.upload_file(local_out, bucket, out_key)
    s3.put_object(Bucket=bucket, Key=suc_key, Body=f"ok target={target}\n".encode("utf-8"))
    print("wrote:", f"s3://{bucket}/{out_key}")
# run for all weeks you care about:
for wk in weeks:
    merge_week(wk, "y_15")
    merge_week(wk, "y_30")

wrote: s3://smart-park-seattle/parking_v2/preds/final/year=2023/target=y_15/week=2023-01-02/pred.csv.gz
wrote: s3://smart-park-seattle/parking_v2/preds/final/year=2023/target=y_30/week=2023-01-02/pred.csv.gz
wrote: s3://smart-park-seattle/parking_v2/preds/final/year=2023/target=y_15/week=2023-01-09/pred.csv.gz
wrote: s3://smart-park-seattle/parking_v2/preds/final/year=2023/target=y_30/week=2023-01-09/pred.csv.gz
wrote: s3://smart-park-seattle/parking_v2/preds/final/year=2023/target=y_15/week=2023-01-16/pred.csv.gz
wrote: s3://smart-park-seattle/parking_v2/preds/final/year=2023/target=y_30/week=2023-01-16/pred.csv.gz
wrote: s3://smart-park-seattle/parking_v2/preds/final/year=2023/target=y_15/week=2023-01-23/pred.csv.gz
wrote: s3://smart-park-seattle/parking_v2/preds/final/year=2023/target=y_30/week=2023-01-23/pred.csv.gz
wrote: s3://smart-park-seattle/parking_v2/preds/final/year=2023/target=y_15/week=2023-01-30/pred.csv.gz
wrote: s3://smart-park-seattle/parking_v2/preds/final/year=2023/

In [6]:
# Sanity Check
import numpy as np

s3 = boto3.client("s3")
bucket, prefix = parse_s3_uri(S3_ROOT)

def read_final(week, target):
    key = f"{prefix}preds/final/year={YEAR}/target={target}/week={week}/pred.csv.gz"
    obj = s3.get_object(Bucket=bucket, Key=key)
    raw = obj["Body"].read()
    with gzip.GzipFile(fileobj=io.BytesIO(raw), mode="rb") as f:
        return pd.read_csv(f)

wk = weeks[0]
df15 = read_final(wk, "y_15")
print(df15.head())
print(df15.columns, df15.shape)
print("p15 range:", df15["p15"].min(), df15["p15"].max())

dup = df15.duplicated(["ts15_utc","sourceelementkey"]).sum()
print("duplicates(ts,node):", dup)

tsu = pd.to_datetime(df15["ts15_utc"], utc=True).sort_values().unique()
deltas = np.diff(tsu[:200]).astype("timedelta64[s]").astype(int)
print("15-min fraction (first 200 deltas):", float((deltas == 900).mean()) if len(deltas) else 1.0)

if "logit15" in df15.columns:
    z = df15["logit15"].to_numpy(dtype=float)
    p = df15["p15"].to_numpy(dtype=float)
    p2 = 1.0 / (1.0 + np.exp(-z))
    print("max|sigmoid(logit15)-p15|:", float(np.max(np.abs(p2 - p))))

                    ts15_utc  sourceelementkey       p15   logit15
0  2023-01-02 10:45:00+00:00              1001  0.735050  1.020395
1  2023-01-02 10:45:00+00:00              1002  0.743804  1.065835
2  2023-01-02 10:45:00+00:00              1005  0.865835  1.864622
3  2023-01-02 10:45:00+00:00              1006  0.702605  0.859733
4  2023-01-02 10:45:00+00:00              1009  0.804035  1.411704
Index(['ts15_utc', 'sourceelementkey', 'p15', 'logit15'], dtype='object') (999432, 4)
p15 range: 4.394719e-12 1.0
duplicates(ts,node): 0
15-min fraction (first 200 deltas): 1.0
max|sigmoid(logit15)-p15|: 1.0000000000287557e-06
